# STAR — Kaggle: train v3c trên 50K (smart-sample + ViTPose, text FROZEN) — RESUME nhiều commit

Y công thức **v3c** (10K → 0.7485), đổi data sang **50K**: `group_by=pair` · **text BERT 0-6 ĐÓNG BĂNG**
(`lora_freeze_text=true`, KHÔNG LoRA text) · **ViTPose** · r=16 · λ_smoothAP=0.2 · lr=2e-4 · LHP=true.
VAL-B split-by-video seed-42 → chọn **best.pth**.

🔴 **GPU/batch:** trainer **CHỈ 1 GPU** (no DDP) → T4×2 thì GPU#2 nằm không, batch theo 1×T4 (15G):
**batch 20** ≈ 13.5G ✅ · **batch 32** ≈ 21G ❌ OOM. Muốn batch lớn → `grad_accum` (không tốn VRAM).

⏱️ **10 epoch trên 50K ≈ 16-19h > 9h** → KHÔNG 1 commit. Giải pháp: **resume nhiều commit.**
Mỗi commit **tự dừng sau 7.5h** (`--max-hours 7.5`) + lưu `last.pth` → **commit kế tiếp chạy tiếp**.
`EPOCHS=10` cố định để lịch LR liền mạch qua các commit.

## Cách chạy nhiều commit (mỗi lần ~8h)
1. **Commit 1**: Add Input = dataset 50K (+ xvlm base). Run All → ~4 epoch → lưu output.
2. **Commit 2**: Add Input **THÊM = Output của Commit 1** (*Add Input → Notebook Output*) → notebook tự tìm `last.pth` → chạy tiếp ~epoch 4-8.
3. **Commit 3**: Add Input = Output của Commit 2 → chạy nốt tới epoch 10.
→ `best.pth` (VAL-B tốt nhất, được mang sang qua các commit) luôn ở output mới nhất.

**GPU T4 · Internet ON.** *(Chạy 1 commit thôi cũng ra best.pth ~4 epoch — đã hơn 10 epoch ở 10K về lượng data.)*

In [ ]:
# [1/7] GPU + clone/pull repo (can ban MOI: PairBatchSampler + FIX bool + resume)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Bat GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert "resume_from" in pathlib.Path("src/star/engine/trainer.py").read_text(), "repo cu — chua co resume, pull lai!"
print("repo OK (co resume):", os.getcwd())

In [ ]:
# [2/7] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/7] Tim + giai nen 50K + base ckpt
import glob, os, pathlib, shutil
def find_one(p):
    h = sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))
    return h[0] if h else None
CKPT = find_one("xvlm_16m_base.th")
ARCH = find_one("train_50k_hard_data.tar.zst")
marker = "/kaggle/working/extract"
got = glob.glob("/kaggle/input/**/train_50k_hard.jsonl", recursive=True) + \
      glob.glob(f"{marker}/**/train_50k_hard.jsonl", recursive=True)
if not got:
    assert ARCH, "Khong thay train_50k_hard_data.tar.zst — Add Input dataset 50K!"
    os.makedirs(marker, exist_ok=True); print("giai nen 50K (~5-8')...")
    if shutil.which("zstd"):
        os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
    else:
        import zstandard, tarfile
        with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
            with tarfile.open(fileobj=zr, mode="r|") as tf:
                tf.extractall(marker)
    got = glob.glob(f"{marker}/**/train_50k_hard.jsonl", recursive=True)
assert got, "khong thay train_50k_hard.jsonl"
DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
print("DATA_ROOT =", DATA_ROOT, "| CKPT =", CKPT,
      "| disk free:", round(shutil.disk_usage('/kaggle/working').free/2**30, 1), "GB")

In [ ]:
# [4/7] Build manifest (GIONG HET v3c: keypoints + bbox + pair_image_id + split video seed-42)
import json, glob, pathlib
import numpy as np, pandas as pd
JSONL = glob.glob(f"{DATA_ROOT}/**/train_*hard.jsonl", recursive=True)[0]
POSEJ = glob.glob(f"{DATA_ROOT}/**/*vitpose*.json", recursive=True)[0]
print("jsonl:", JSONL, "\nvitpose:", POSEJ)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")
for line in open(JSONL, encoding="utf-8"):
    r = json.loads(line); anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)
df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(POSEJ, encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid); b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of); df["bbox"] = df.image_id.map(bbox_of)
rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""
MANIFEST = "/kaggle/working/manifest_50k_hard.parquet"; df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} kpts={df.keypoints.notna().mean():.0%} leakage={leak}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/7] CAU HINH v3c (EPOCHS=10 co dinh; moi commit tu dung sau 7.5h)
BATCH = 20            # single-GPU -> 20 (~13.5G). 32 = OOM. Batch lon -> tang grad_accum.
EPOCHS = 10           # muc tieu day du; --max-hours cat moi commit cho <9h
MAX_HOURS = 7.5       # moi commit chay <=7.5h roi luu last.pth + dung (tong <9h voi setup)
import pandas as pd
dfm = pd.read_parquet(MANIFEST)
PAIRS = int(dfm[dfm.split == "train"].pair_image_id.notna().sum())
epm = (PAIRS / max(BATCH // 2, 1)) * 1.6 / 60
print(f"pairs(train)={PAIRS} ~{epm:.0f} min/epoch -> ~{MAX_HOURS*60/epm:.1f} epoch/commit "
      f"-> ~{int(np.ceil(EPOCHS/(MAX_HOURS*60/epm)))} commit cho du {EPOCHS} epoch")
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"
V3C = (f"data.group_by=pair model.lora_freeze_text=true model.pose_enabled=true "
       f"data.lhp_enabled=true loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
       f"optim.epochs={EPOCHS} train.early_stop_patience=8 train.eval_every_epochs=0.5 "
       f"train.batch_size={BATCH} train.grad_accum=2 train.grad_norm_every=0 train.out_dir=/kaggle/working/v3c50k")
print("CONFIG:", V3C)

In [ ]:
# [6/7] GATE overfit-one-batch (probe vram). OOM -> ha BATCH=16
gate = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch --set {DATA} {V3C}"
print(gate)
!{gate}

In [ ]:
# [7/7] TRAIN: tu tim checkpoint commit truoc (resume) + tu dung 7.5h -> best.pth
import glob, shutil, pathlib, time, torch
pathlib.Path("/kaggle/working/v3c50k").mkdir(parents=True, exist_ok=True)
# checkpoint cua commit TRUOC (neu da Add Input = Notebook Output cua commit truoc)
prev_last = sorted(glob.glob("/kaggle/input/**/v3c50k/last.pth", recursive=True))
prev_best = sorted(glob.glob("/kaggle/input/**/v3c50k/best.pth", recursive=True))
RESUME = prev_last[0] if prev_last else None
if prev_best:                                  # mang best.pth chay tot nhat sang commit nay
    shutil.copy(prev_best[0], "/kaggle/working/v3c50k/best.pth")
print("RESUME:", RESUME or "(lan dau - train tu epoch 0)")

resume_arg = f"--resume {RESUME}" if RESUME else ""
cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml "
       f"--max-hours {MAX_HOURS} {resume_arg} --set {DATA} {V3C}")
print(cmd); t0 = time.time()
!{cmd}
mins = round((time.time() - t0) / 60, 1)
BEST = "/kaggle/working/v3c50k/best.pth"
assert pathlib.Path(BEST).exists(), "train fail — xem log (OOM? ha BATCH=16)"
raw = torch.load(BEST, map_location="cpu", weights_only=False)
print("=" * 70)
print(f"COMMIT xong ({mins}'): VAL-B best mAP = {raw.get('best_metric'):.4f} (moc v3c-10K=0.7485)")
print(f"  report: {(raw.get('extra') or {}).get('report')}")
print("  -> Neu chua du epoch: commit moi, Add Input = Output commit nay, Run All de chay tiep.")
del raw
!ls -lh /kaggle/working/v3c50k/

## Lấy kết quả
- **best.pth** ở `/kaggle/working/v3c50k/best.pth` (Output tab). VAL-B mAP in ở cell [7].
- **Chưa đủ 10 epoch?** Log sẽ ghi `[time-stop] ... dung de commit kip <9h`. → **commit mới**, *Add Input → Notebook Output* của commit vừa rồi → **Run All** → tự resume chạy tiếp. Lặp ~3 lần là đủ 10 epoch.
- best.pth được **mang sang** qua các commit (giữ bản VAL-B tốt nhất). last.pth = trạng thái để resume.
- **batch 20** (32 OOM, T4×2 không giúp train vì 1-GPU). Muốn eff. batch lớn → tăng `grad_accum`.
- **1 commit thôi** cũng ra best.pth ~4 epoch (đã nhiều data hơn 10 epoch ở 10K).